# 🚀 Модуль 2. Урок 6. Промышленные паттерны SQL: Когортный анализ (Retention) и воронки конверсий

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_02_sql/lesson_06_sql_patterns_retention_funnels.ipynb)

## 📌 Бизнес-контекст и методология
В продуктовой аналитике SQL используется для расчета когортного удержания (Retention) и воронок конверсий.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

db_path = ensure_dataset("ecommerce.db")
conn = sqlite3.connect(db_path)
print(f"🟢 Подключение к базе данных успешно установлено: {db_path}")


🟢 Подключение к базе данных успешно установлено: course\datasets\ecommerce.db


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(ensure_dataset("ecommerce.db"))

query = """
WITH CustomerCohorts AS (
    SELECT 
        customer_id,
        STRFTIME('%Y-%m', MIN(order_date)) AS cohort_month
    FROM orders
    WHERE status = 'Completed'
    GROUP BY customer_id
),
OrderActivity AS (
    SELECT 
        o.customer_id,
        cc.cohort_month,
        STRFTIME('%Y-%m', o.order_date) AS activity_month,
        (CAST(STRFTIME('%Y', o.order_date) AS INTEGER) - CAST(SUBSTR(cc.cohort_month, 1, 4) AS INTEGER)) * 12 +
        (CAST(STRFTIME('%m', o.order_date) AS INTEGER) - CAST(SUBSTR(cc.cohort_month, 6, 2) AS INTEGER)) AS period_month
    FROM orders o
    JOIN CustomerCohorts cc ON o.customer_id = cc.customer_id
    WHERE o.status = 'Completed'
)
SELECT 
    cohort_month,
    period_month,
    COUNT(DISTINCT customer_id) AS active_users
FROM OrderActivity
GROUP BY cohort_month, period_month
ORDER BY cohort_month, period_month;
"""

df_cohorts = pd.read_sql_query(query, conn)
cohort_matrix = df_cohorts.pivot(index='cohort_month', columns='period_month', values='active_users')
cohort_matrix.head(6)


period_month,0,1,2,3,4,5,6,7,8,9
cohort_month,,,,,,,,,,
2025-01,19.0,5.0,2.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN
2025-02,44.0,16.0,9.0,7.0,3.0,NaN,NaN,NaN,NaN,NaN
2025-03,58.0,18.0,15.0,9.0,8.0,3.0,2.0,2.0,NaN,NaN
2025-04,62.0,19.0,9.0,5.0,2.0,1.0,2.0,NaN,NaN,NaN
2025-05,57.0,27.0,13.0,6.0,6.0,4.0,2.0,1.0,1.0,1.0
2025-06,57.0,19.0,10.0,5.0,6.0,3.0,1.0,1.0,1.0,1.0


### 🎯 Практическое задание 1: Расчет Retention Rate %
Рассчитайте процент удержания когорт относительно начального периода.

In [3]:
### ВАШ КОД ЗДЕСЬ
cohort_size = cohort_matrix.iloc[:, 0]
retention_rate = cohort_matrix.divide(cohort_size, axis=0) * 100

print("Матрица удержания (%):")
print(retention_rate.round(1).head(5))

assert retention_rate.iloc[:, 0].max() == 100.0, "В нулевой месяц удержание всегда 100%"
print("✅ Тест успешно пройден: продуктовая матрица Retention сформирована!")


Матрица удержания (%):
period_month      0     1     2     3     4    5    6    7    8    9
cohort_month                                                        
2025-01       100.0  26.3  10.5  15.8   NaN  NaN  NaN  NaN  NaN  NaN
2025-02       100.0  36.4  20.5  15.9   6.8  NaN  NaN  NaN  NaN  NaN
2025-03       100.0  31.0  25.9  15.5  13.8  5.2  3.4  3.4  NaN  NaN
2025-04       100.0  30.6  14.5   8.1   3.2  1.6  3.2  NaN  NaN  NaN
2025-05       100.0  47.4  22.8  10.5  10.5  7.0  3.5  1.8  1.8  1.8
✅ Тест успешно пройден: продуктовая матрица Retention сформирована!


### 💡 Формула воронки в SQL
Используйте `COUNT(CASE WHEN event = 'checkout' THEN user_id END)` для быстрого построения воронки без тяжелых JOIN.